# Lab06: Movie Rating Completion
### Hunter Shook and Drew Schipper


### AI Notice
##### AI was used to address the following questions:
* 

In [59]:
import pandas as pd

def create_user_movie_matrix():
    """
    Loads ratings and movie data, then creates a user-movie ratings matrix.
    The resulting matrix has user IDs as rows, movie titles as columns,
    and user ratings as the values.

    Returns:
        pd.DataFrame: A pivot table representing the user-movie ratings matrix.
                      Ratings are filled with NaN where a user has not rated a movie.
    """
    try:
        # Load the ratings and movies CSV files into DataFrames
        ratings = pd.read_csv('ratings.csv')
        movies = pd.read_csv('movies.csv')

        # Drop the 'timestamp' column from the ratings DataFrame as it's not needed
        ratings = ratings.drop('timestamp', axis=1)

        # Merge the ratings and movies DataFrames on the 'movieId' column
        # This adds the movie titles to the ratings data
        movie_ratings = pd.merge(ratings, movies, on='movieId')

        # Create the pivot table (user-movie matrix)
        # index='userId' -> Rows will be user IDs
        # columns='title' -> Columns will be movie titles
        # values='rating' -> The values in the table will be the ratings
        # This will automatically handle duplicate entries by averaging ratings,
        # but for this dataset, there's only one rating per user-movie pair.
        user_movie_matrix = movie_ratings.pivot_table(
            index='userId',
            columns='title',
            values='rating'
        )

        return user_movie_matrix

    except FileNotFoundError as e:
        print(f"Error: One of the required files was not found. Please ensure 'ratings.csv' and 'movies.csv' are in the same directory.")
        print(e)
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

if __name__ == '__main__':
    user_movie_ratings_matrix = create_user_movie_matrix()

    if user_movie_ratings_matrix is not None:
        print("Successfully created the user-movie ratings matrix.")
        print("\nDisplaying the first 5 rows and 5 columns of the matrix:")
        # Displaying a small part of the matrix for demonstration
        print(user_movie_ratings_matrix.iloc[:5, :5])
        print("\nMatrix shape: ", user_movie_ratings_matrix.shape)
        print("Total number of movies rated by at least one user: ", user_movie_ratings_matrix.shape[1])
        print("Total number of users who rated at least one movie: ", user_movie_ratings_matrix.shape[0])


Successfully created the user-movie ratings matrix.

Displaying the first 5 rows and 5 columns of the matrix:
title   '71 (2014)  'Hellboy': The Seeds of Creation (2004)  \
userId                                                        
1              NaN                                      NaN   
2              NaN                                      NaN   
3              NaN                                      NaN   
4              NaN                                      NaN   
5              NaN                                      NaN   

title   'Round Midnight (1986)  'Salem's Lot (2004)  'Til There Was You (1997)  
userId                                                                          
1                          NaN                  NaN                        NaN  
2                          NaN                  NaN                        NaN  
3                          NaN                  NaN                        NaN  
4                          NaN             

In [60]:
import pandas as pd
from sklearn.model_selection import train_test_split

def create_user_movie_matrix():
    """
    Loads ratings and movie data, then creates a user-movie ratings matrix.
    The resulting matrix has user IDs as rows, movie titles as columns,
    and user ratings as the values.

    Returns:
        tuple: A tuple containing two pandas DataFrames:
               (train_matrix, test_matrix)
    """
    try:
        # Load the ratings and movies CSV files into DataFrames
        ratings = pd.read_csv('ratings.csv')
        movies = pd.read_csv('movies.csv')

        # Drop the 'timestamp' column from the ratings DataFrame as it's not needed
        ratings = ratings.drop('timestamp', axis=1)

        # Merge the ratings and movies DataFrames on the 'movieId' column
        # This adds the movie titles to the ratings data
        movie_ratings = pd.merge(ratings, movies, on='movieId')

        # Count the number of ratings per user and per movie
        user_counts = movie_ratings['userId'].value_counts()
        movie_counts = movie_ratings['title'].value_counts()

        # Filter users who have rated at least 5 movies
        users_to_keep = user_counts[user_counts >= 5].index
        filtered_ratings = movie_ratings[movie_ratings['userId'].isin(users_to_keep)]

        # Filter movies that have been rated at least 15 times
        movies_to_keep = movie_counts[movie_counts >= 15].index
        final_filtered_ratings = filtered_ratings[filtered_ratings['title'].isin(movies_to_keep)]
        
        # Split the filtered ratings DataFrame into a training set (80%) and a testing set (20%)
        # All users and movies have a chance to be in both sets

        #this keeps all the rows and columns which gives us the distribution/model of testing and training we are looking for. All users should be present in both sets
        train_df, test_df = train_test_split(final_filtered_ratings, test_size=0.2, random_state=42)

        # Create the pivot table for the training data
        train_matrix = train_df.pivot_table(
            index='userId',
            columns='title',
            values='rating'
        )

        # Create the pivot table for the testing data
        test_matrix = test_df.pivot_table(
            index='userId',
            columns='title',
            values='rating'
        )
        
        return train_matrix, test_matrix

    except FileNotFoundError as e:
        print(f"Error: One of the required files was not found. Please ensure 'ratings.csv' and 'movies.csv' are in the same directory.")
        print(e)
        return None, None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None, None

if __name__ == '__main__':
    train_matrix, test_matrix = create_user_movie_matrix()

    if train_matrix is not None and test_matrix is not None:
        print("Successfully created the training and testing matrices.")
        
        print("\nTraining matrix shape: ", train_matrix.shape)
        print("Testing matrix shape: ", test_matrix.shape)

        print("\nDisplaying the first 5 rows and 5 columns of the training matrix:")
        print(train_matrix.iloc[:5, :5])
        
        print("\nDisplaying the first 5 rows and 5 columns of the testing matrix:")
        print(test_matrix.iloc[:5, :5])


Successfully created the training and testing matrices.

Training matrix shape:  (611, 1650)
Testing matrix shape:  (610, 1640)

Displaying the first 5 rows and 5 columns of the training matrix:
title   'burbs, The (1989)  (500) Days of Summer (2009)  \
userId                                                    
1                      NaN                          NaN   
2                      NaN                          NaN   
3                      NaN                          NaN   
4                      NaN                          NaN   
5                      NaN                          NaN   

title   10 Things I Hate About You (1999)  10,000 BC (2008)  \
userId                                                        
1                                     NaN               NaN   
2                                     NaN               NaN   
3                                     NaN               NaN   
4                                     NaN               NaN   
5            

In [61]:
import pandas as pd
from sklearn.model_selection import train_test_split

def create_user_movie_matrix():
    """
    Loads ratings and movie data, then creates a user-movie ratings matrix.
    The resulting matrix has user IDs as rows, movie titles as columns,
    and user ratings as the values.

    Returns:
        tuple: A tuple containing two pandas DataFrames:
               (train_matrix, test_matrix)
    """
    try:
        # Load the ratings and movies CSV files into DataFrames
        ratings = pd.read_csv('ratings.csv')
        movies = pd.read_csv('movies.csv')

        # Drop the 'timestamp' column from the ratings DataFrame as it's not needed
        ratings = ratings.drop('timestamp', axis=1)

        # Merge the ratings and movies DataFrames on the 'movieId' column
        # This adds the movie titles to the ratings data
        movie_ratings = pd.merge(ratings, movies, on='movieId')

        # Count the number of ratings per user and per movie
        user_counts = movie_ratings['userId'].value_counts()
        movie_counts = movie_ratings['title'].value_counts()

        # Filter users who have rated at least 5 movies
        users_to_keep = user_counts[user_counts >= 5].index
        filtered_ratings = movie_ratings[movie_ratings['userId'].isin(users_to_keep)]

        # Filter movies that have been rated at least 15 times
        movies_to_keep = movie_counts[movie_counts >= 15].index
        final_filtered_ratings = filtered_ratings[filtered_ratings['title'].isin(movies_to_keep)]
        
        # Split the filtered ratings DataFrame into a training set (80%) and a testing set (20%)
        # All users and movies have a chance to be in both sets
        train_df, test_df = train_test_split(final_filtered_ratings, test_size=0.2, random_state=42)

        # Create the pivot table for the training data
        train_matrix = train_df.pivot_table(
            index='userId',
            columns='title',
            values='rating'
        )

        # Create the pivot table for the testing data
        test_matrix = test_df.pivot_table(
            index='userId',
            columns='title',
            values='rating'
        )
        
        return train_matrix, test_matrix

    except FileNotFoundError as e:
        print(f"Error: One of the required files was not found. Please ensure 'ratings.csv' and 'movies.csv' are in the same directory.")
        print(e)
        return None, None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None, None

if __name__ == '__main__':
    train_matrix, test_matrix = create_user_movie_matrix()

    if train_matrix is not None and test_matrix is not None:
        print("Successfully created the training and testing matrices.")
        
        print("\nTraining matrix shape: ", train_matrix.shape)
        print("Testing matrix shape: ", test_matrix.shape)

        print("\nDisplaying the first 5 rows and 5 columns of the training matrix:")
        print(train_matrix.iloc[:5, :5])
        
        print("\nDisplaying the first 5 rows and 5 columns of the testing matrix:")
        print(test_matrix.iloc[:5, :5])


Successfully created the training and testing matrices.

Training matrix shape:  (611, 1650)
Testing matrix shape:  (610, 1640)

Displaying the first 5 rows and 5 columns of the training matrix:
title   'burbs, The (1989)  (500) Days of Summer (2009)  \
userId                                                    
1                      NaN                          NaN   
2                      NaN                          NaN   
3                      NaN                          NaN   
4                      NaN                          NaN   
5                      NaN                          NaN   

title   10 Things I Hate About You (1999)  10,000 BC (2008)  \
userId                                                        
1                                     NaN               NaN   
2                                     NaN               NaN   
3                                     NaN               NaN   
4                                     NaN               NaN   
5            

# Rank 65

In [62]:
import numpy as np



N = train_matrix.shape[0]  # Number of users (rows)
M = train_matrix.shape[1]  # Number of movies (columns)
K = 65                     # Desired Rank/Number of Latent Features

np.random.seed(42)

# Calculate the standard deviation for initialization, used to keep within range
std_dev = 1 / np.sqrt(K)

# Initialize P (N users x K features)
# loc=0.0 sets the mean to 0
P = np.random.normal(loc=0.0, scale=std_dev, size=(N, K))

# Initialize Q (M movies x K features)
# Q.T will be used in the prediction R' = P @ Q.T
Q = np.random.normal(loc=0.0, scale=std_dev, size=(M, K))

print("\n🚀 P and Q matrices initialized successfully.")
print(f"   P matrix shape (Users x Rank): {P.shape}")
print(f"   Q matrix shape (Movies x Rank): {Q.shape}")


🚀 P and Q matrices initialized successfully.
   P matrix shape (Users x Rank): (611, 65)
   Q matrix shape (Movies x Rank): (1650, 65)


# first RMSE

In [63]:
# need to loop and create the training curve...error value vs. epoch/training time

PQ = P @ Q.T

#back into pandas data frame for easy comparison
PQ = pd.DataFrame(
    PQ,
    index=train_matrix.index,
    columns=train_matrix.columns
)

# Extract the actual (true) ratings from the training set.
# .stack() converts the sparse matrix into a Series of (userId, title, rating) tuples,
# ignoring the NaN values.
actual_ratings = train_matrix.stack()

# Extract the predicted ratings for the exact same (user, movie) pairs.
# R_prime_df.stack() will have the same index as actual_ratings.
predicted_ratings = PQ.stack()

#actual and training data is now ready
#sum of differences squared
squared_errors = (actual_ratings - predicted_ratings) ** 2
rmse = np.sqrt(squared_errors.mean())

print(rmse)

3.7427696096791405


In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_training_curve(train_rmse_history, test_rmse_history):
    """Generates and displays the RMSE vs. Epoch plot, overlaying test data."""
    
    if len(train_rmse_history) != len(test_rmse_history):
        print("Error: Train and Test history lists must be the same length.")
        return

    epochs = range(1, len(train_rmse_history) + 1)
    
    plt.figure(figsize=(10, 6))
    
    # Plot Training RMSE
    plt.plot(epochs, train_rmse_history, marker='o', linestyle='-', color='b', label='Training RMSE')
    
    # Plot Test RMSE
    plt.plot(epochs, test_rmse_history, marker='s', linestyle='--', color='r', label='Test RMSE')
    
    plt.title('Matrix Factorization Convergence (Train vs. Test RMSE)')
    plt.xlabel('Epoch Value')
    plt.ylabel('RMSE (Rating Error)')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(epochs) 
    plt.legend() # Show the legend to identify the lines
    
    # --- CHANGE THIS LINE ---
    # The plot will appear automatically because of '%matplotlib inline',
    # but plt.show() ensures immediate rendering if the inline command is used.
    plt.show() 
    
    # REMOVED: plt.savefig(...)
    # REMOVED: plt.close()
    print("\n✅ Convergence curve displayed inline.")


#test data
test_ratings = test_matrix.stack()

# Hyperparameters
step_rate = 0.01  # gamma (γ)
num_epochs = 80       # Number of passes over the training data

# 1. Get a list of all known (i, j, rating) triplets from train_matrix...this is vital to be able to step through all the "real"/present data points
training_triplets = actual_ratings.reset_index()
test_triplets = test_ratings.reset_index()
rmse_history = [] 
test_rmse_history = []

for epoch in range(num_epochs):
    PQ = P @ Q.T

    #back into pandas data frame for easy comparison
    PQ = pd.DataFrame(
        PQ,
        index=train_matrix.index,
        columns=train_matrix.columns
    )
    
    # Extract the actual (true) ratings from the training set.
    # .stack() converts the sparse matrix into a Series of (userId, title, rating) tuples,
    # ignoring the NaN values.
    actual_ratings = train_matrix.stack()
    
    # Extract the predicted ratings for the exact same (user, movie) pairs.
    # R_prime_df.stack() will have the same index as actual_ratings.
    predicted_ratings = PQ.stack()
    
    #actual and training data is now ready
    #sum of differences squared
    squared_errors = (actual_ratings - predicted_ratings) ** 2
    rmse = np.sqrt(squared_errors.mean())

    #rmse for TEST
    t_squared_errors = (test_ratings - predicted_ratings) ** 2
    t_rmse = np.sqrt(t_squared_errors.mean())
    
    print(rmse)
    
    rmse_history.append(rmse)
    test_rmse_history.append(t_rmse)
    
    # Iterate over every known rating in the training set
    for index, row in training_triplets.iterrows():
        user_id = row['userId']
        movie_title = row['title']
        actual_rating = row[0] # The rating value itself

        # Get the row/column indices from the training 
        i_idx = train_matrix.index.get_loc(user_id)
        j_idx = train_matrix.columns.get_loc(movie_title)

        # Get the current factor vectors from P and Q
        p_i = P[i_idx, :]
        q_j = Q[j_idx, :]

        # Calculate prediction and error (Step 1)
        prediction = p_i @ q_j.T # or p_i @ q_j.T
        error = actual_rating - prediction

        # Note: p_i and q_j are updated in place since they are slices/views of P and Q
        P[i_idx, :] = p_i + (step_rate * q_j * error)
        Q[j_idx, :] = q_j + (step_rate * p_i * error)
        
plot_training_curve(rmse_history, test_rmse_history)

3.7427696096791405
2.425093719550083
1.1475848851508625
0.8738248228347173
0.7871262124345121
0.7395463215396726
0.7031510694799368
0.6700967880220889


KeyboardInterrupt: 

### get P and Q back to origional random

### Get the P and Q matrices we want to predict our data

In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N = train_matrix.shape[0]  # Number of users (rows)
M = train_matrix.shape[1]  # Number of movies (columns)
K = 65                     # Desired Rank/Number of Latent Features

np.random.seed(42)

# Calculate the standard deviation for initialization, used to keep within range
std_dev = 1 / np.sqrt(K)

# Initialize P (N users x K features)
# loc=0.0 sets the mean to 0
P2 = np.random.normal(loc=0.0, scale=std_dev, size=(N, K))

# Initialize Q (M movies x K features)
# Q.T will be used in the prediction R' = P @ Q.T
Q2 = np.random.normal(loc=0.0, scale=std_dev, size=(M, K))

#test data
test_ratings = test_matrix.stack()

# Hyperparameters
step_rate = 0.01  # gamma (γ)
num_epochs = 6       # Number of passes over the training data

# 1. Get a list of all known (i, j, rating) triplets from train_matrix...this is vital to be able to step through all the "real"/present data points
training_triplets = actual_ratings.reset_index()
test_triplets = test_ratings.reset_index()
rmse_history = [] 
test_rmse_history = []

for epoch in range(num_epochs):
    PQ2 = P2 @ Q2.T

    #back into pandas data frame for easy comparison
    PQ2 = pd.DataFrame(
        PQ2,
        index=train_matrix.index,
        columns=train_matrix.columns
    )
    
    # Extract the actual (true) ratings from the training set.
    # .stack() converts the sparse matrix into a Series of (userId, title, rating) tuples,
    # ignoring the NaN values.
    actual_ratings = train_matrix.stack()
    
    # Extract the predicted ratings for the exact same (user, movie) pairs.
    # R_prime_df.stack() will have the same index as actual_ratings.
    predicted_ratings = PQ2.stack()
    
    #actual and training data is now ready
    #sum of differences squared
    squared_errors = (actual_ratings - predicted_ratings) ** 2
    rmse = np.sqrt(squared_errors.mean())

    #rmse for TEST
    t_squared_errors = (test_ratings - predicted_ratings) ** 2
    t_rmse = np.sqrt(t_squared_errors.mean())
    
    print(rmse)
    
    rmse_history.append(rmse)
    test_rmse_history.append(t_rmse)
    
    # Iterate over every known rating in the training set
    for index, row in training_triplets.iterrows():
        user_id = row['userId']
        movie_title = row['title']
        actual_rating = row[0] # The rating value itself

        # Get the row/column indices from the training 
        i_idx = train_matrix.index.get_loc(user_id)
        j_idx = train_matrix.columns.get_loc(movie_title)

        # Get the current factor vectors from P and Q
        p_i = P2[i_idx, :]
        q_j = Q2[j_idx, :]

        # Calculate prediction and error (Step 1)
        prediction = p_i @ q_j.T # or p_i @ q_j.T
        error = actual_rating - prediction

        # Note: p_i and q_j are updated in place since they are slices/views of P and Q
        P2[i_idx, :] = p_i + (step_rate * q_j * error)
        Q2[j_idx, :] = q_j + (step_rate * p_i * error)

user_id = 611

predictions_array = PQ2.values
MIN_RATING = 0.5
MAX_RATING = 5.0

clipped_predictions_array = np.clip(
    predictions_array,
    a_min=MIN_RATING,
    a_max=MAX_RATING
)

# 3. Create a new DataFrame using the clipped values
R_prime_clipped_df = pd.DataFrame(
    clipped_predictions_array,
    index=PQ2.index,
    columns=PQ2.columns
)
# 1. Get the user's predicted ratings
user_predictions = R_prime_clipped_df.loc[user_id]

# 2. Get the user's actual ratings (to find out what they haven't seen)
user_actual_ratings = train_matrix.loc[user_id]

# 3. Filter predictions to only include movies the user has NOT rated (where actual rating is NaN)
unrated_predictions = user_predictions[user_actual_ratings.isna()]

# 4. Sort and get the top 10 recommendations
top_10_recommendations = unrated_predictions.sort_values(ascending=False).head(10)

print(f"\nTop 10 Movie Recommendations for User {user_id}:")
print(top_10_recommendations)

3.7427696096791405
2.425093719550083
1.1475848851508625
0.8738248228347173
0.7871262124345121
0.7395463215396726

Top 10 Movie Recommendations for User 611:
title
Pulp Fiction (1994)                  3.287387
Shawshank Redemption, The (1994)     3.021233
Fugitive, The (1993)                 2.924299
Silence of the Lambs, The (1991)     2.902432
Toy Story (1995)                     2.890777
Matrix, The (1999)                   2.863654
Forrest Gump (1994)                  2.850953
Braveheart (1995)                    2.810141
Seven (a.k.a. Se7en) (1995)          2.800131
Die Hard: With a Vengeance (1995)    2.796070
Name: 611, dtype: float64
